# [A-CP7] Top-10 incumbent repeat winners and displacement exposure

**Status:** exploratory, non-citable
**Research question:** [A-CP7 — New-entrant vs. repeat-winner mix](../../docs/research-questions.md)
**Decision this informs:** whether award *concentration among frequent winners* is large enough
to treat as a crowd-out *proxy* for first-time and low-award-volume firms. Not a causal
crowd-out estimate.
**Data as of:** local `data/raw/sbir/award_data.csv` (mtime 2026-05-11)
**Owner:** exploratory workbench

Descriptive question: in each fiscal year *t*, how much Phase I award capacity is absorbed
by the ten firms with the most awards *through t−1*, and how many additional one-award
firms could mechanically fit if those incumbents received only one Phase I award?

This is **not** causal crowd-out. The corpus is awards to winners. It does not contain
applicants who lost. Results are **displacement exposure / reallocation capacity /
crowd-out proxies**. Two awards in the same agency-year-phase need not have competed
for the same solicitation.

Do not read "low-volume incumbent" as "small firm." SBIR/STTR recipients are program
small businesses at award time; volume here is *prior award count*, not employment.


## Data contract

- **Population:** Public SBIR.gov SBIR/STTR awards after `load_sbir_awards_csv` edition
  collapse (`award_key` grain). Phase I is the headline outcome; Phase II is separate.
  Phase III is absent from this extract.
- **Grain:** one row per collapsed `award_key`. Firm grain is `organization_id` from
  `dod_supply_chain_baseline._identity_frame` (UEI, then DUNS, then normalized name).
- **Keys:** `award_key` (award), `organization_id` (firm), `analysis_year` (federal FY from
  Proposal Award Date when ≥1983, else SBIR.gov Award Year).
- **Inputs:** `data/raw/sbir/award_data.csv`. CET classifications are **not** present
  locally (the public `cet` column is empty; no corpus-wide classification parquet).
- **Missingness:** missing Proposal Award Date (~48.5%) is filled from Award Year, not
  treated as a non-award. Missing solicitation/topic is a coverage gap for the
  within-opportunity slice. Name-only identity is a lower-confidence firm key.
- **Outputs:** this notebook only. Exploratory. No canonical generator, no study contract.
- **Look-ahead:** incumbent top 10 in year *t* are ranked on awards through *t−1* only.
  A retrospective FY1983–FY2025 top 10 is an appendix, not the headline ranking.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPO_ROOT


In [ ]:
AS_OF_DATE = "2026-05-11"  # mtime of data/raw/sbir/award_data.csv
RANDOM_SEED = 20260819
HEADLINE_YEARS = range(2021, 2026)
COVERAGE_START = 1983  # first year with continuous SBIR.gov Award Year volume
LOOKBACK_PRIMARY = 10
LOOKBACK_ALT = 5
LOW_VOLUME_MAX = 5
TOP_N = 10
MIN_POOL = 30  # agency-year Phase I pools smaller than this stay in tables, drop from panel

REQUIRED_INPUTS = {
    "awards": REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv",
}
OPTIONAL_INPUTS = {
    "enriched_awards": REPO_ROOT / "data" / "processed" / "enriched_sbir_awards.parquet",
    "cet_classifications": REPO_ROOT / "data" / "processed" / "cet_classifications.parquet",
}


In [ ]:
input_status = pd.DataFrame(
    [
        {
            "input": name,
            "path": str(path.relative_to(REPO_ROOT)),
            "exists": path.exists(),
            "role": "required",
        }
        for name, path in REQUIRED_INPUTS.items()
    ]
    + [
        {
            "input": name,
            "path": str(path.relative_to(REPO_ROOT)),
            "exists": path.exists(),
            "role": "optional / missing locally",
        }
        for name, path in OPTIONAL_INPUTS.items()
    ]
)
input_status


## Exploration

Reuse, do not fork:

- `load_sbir_awards_csv` — public CSV → `award_key` / `award_id` identity
- `_identity_frame`, `_federal_fiscal_years`, `_normalize_phase`, `_hhi`,
  `_representative_organization_name` — same firm key and concentration helpers as the
  A-CP7 DoD CET baseline (`sbir_etl.reporting.dod_supply_chain_baseline`)
- `sbir_etl.identity.normalize_company_name` is the primitive inside name fallback via
  that baseline's existing path; this notebook does not add a second normalizer

A-CP7's published DoD CET note ranks concentration *inside classified CET areas* on a
look-ahead window. This pass is program-wide, year-by-year incumbent ranking, Phase I
headline, and crowd-out **proxies**.


In [ ]:
from sbir_etl.extractors.sbir_public_awards import load_sbir_awards_csv
from sbir_etl.reporting.dod_supply_chain_baseline import (
    _federal_fiscal_years,
    _hhi,
    _identity_frame,
    _normalize_phase,
    _representative_organization_name,
)

raw = load_sbir_awards_csv(REQUIRED_INPUTS["awards"])
frame = raw.rename(
    columns={
        "company": "company_name",
        "uei": "company_uei",
        "duns": "company_duns",
        "amount": "award_amount",
    }
)
fy = _federal_fiscal_years(frame)
fy = fy.where(fy >= COVERAGE_START)
award_year = pd.to_numeric(frame["award_year"], errors="coerce")
frame["analysis_year"] = fy.fillna(award_year).astype("Int64")
frame["phase_n"] = frame["phase"].map(_normalize_phase)
identity = _identity_frame(frame)
for col in identity.columns:
    frame[col] = identity[col].to_numpy()
unresolved = frame["organization_id"].isna()
frame.loc[unresolved, "organization_id"] = frame.loc[unresolved, "award_key"].map(
    lambda value: f"award:{value}"
)
frame["program_n"] = frame["program"].fillna("").astype(str).str.strip().str.upper()
frame["solicitation"] = frame["solicitation_number"].fillna("").astype(str).str.strip()
frame["topic"] = frame["topic_code"].fillna("").astype(str).str.strip()
frame["has_solicitation"] = frame["solicitation"].ne("")
frame["has_topic"] = frame["topic"].ne("")
frame = frame.loc[frame["analysis_year"].notna()].copy()
frame["analysis_year"] = frame["analysis_year"].astype(int)
frame["award_amount"] = pd.to_numeric(frame["award_amount"], errors="coerce")

coverage = pd.DataFrame(
    {
        "metric": [
            "rows after edition collapse",
            "analysis_year min/max",
            "share using Award Year fallback",
            "Phase I / II",
            "SBIR / STTR",
            "UEI or DUNS (exact identity)",
            "solicitation number present",
            "topic code present",
            "CET column non-empty",
        ],
        "value": [
            f"{len(frame):,}",
            f"{frame['analysis_year'].min()}–{frame['analysis_year'].max()}",
            f"{float(fy.isna().mean()):.1%}",
            f"{int((frame['phase_n']=='I').sum()):,} / {int((frame['phase_n']=='II').sum()):,}",
            f"{int((frame['program_n']=='SBIR').sum()):,} / {int((frame['program_n']=='STTR').sum()):,}",
            f"{float(frame['exact_identity'].mean()):.1%}",
            f"{float(frame['has_solicitation'].mean()):.1%}",
            f"{float(frame['has_topic'].mean()):.1%}",
            f"{float((raw['cet'].fillna('').astype(str).str.strip()!='').mean()):.1%}",
        ],
    }
)
coverage


### Year axis and left-censoring

SBIR.gov Award Year is populated from 1983. Proposal Award Date (true federal FY) is
sparse before ~2004 and missing on about half of all rows. When both exist they agree
on 95.5% of rows.

**Rule:** `analysis_year` = federal FY from Proposal Award Date if that FY is ≥1983,
else Award Year. Headline window FY2021–FY2025 uses mostly dated rows.

A new entrant in year *t* is a resolved firm with **zero** observed awards before *t*
in this corpus. The public extract does not see pre-1983 awards (SBIR started 1982, so
the remaining left-censor is mostly **identity splits**, not missing program years).

Reliable-entrant flags still require *t* ≥ coverage start + lookback (5y → 1988, 10y →
1993). The headline window satisfies both.


In [ ]:
def firm_history_through(facts: pd.DataFrame, year: int) -> pd.DataFrame:
    prior = facts.loc[facts["analysis_year"] < year]
    if prior.empty:
        return pd.DataFrame(
            columns=["prior_awards", "prior_dollars", "organization_name", "identity_method", "exact_identity"]
        )
    grouped = prior.groupby("organization_id", dropna=False)
    out = grouped.agg(
        prior_awards=("award_key", "nunique"),
        prior_dollars=("award_amount", "sum"),
        identity_method=("identity_method", lambda s: s.value_counts().index[0]),
        exact_share=("exact_identity", "mean"),
    )
    out["organization_name"] = grouped["organization_name"].apply(_representative_organization_name)
    out["exact_identity"] = out["exact_share"] >= 0.5
    return out.drop(columns=["exact_share"])


def top_n_ids(history: pd.DataFrame, *, n: int = TOP_N, rank: str = "prior_awards") -> list[str]:
    if history.empty:
        return []
    ordered = history.sort_values(
        [rank, "prior_dollars", "organization_id"],
        ascending=[False, False, True],
        kind="mergesort",
    )
    return list(ordered.head(n).index)


def volume_bucket(prior_awards: float, is_top10: bool) -> str:
    if is_top10:
        return "top10"
    prior = int(prior_awards)
    if prior <= 0:
        return "entrant"
    if prior == 1:
        return "1"
    if prior <= 5:
        return "2-5"
    if prior <= 20:
        return "6-20"
    return "21+"


def annotate_year(
    facts: pd.DataFrame,
    year: int,
    *,
    rank: str = "prior_awards",
    history_facts: pd.DataFrame | None = None,
) -> pd.DataFrame:
    source = facts if history_facts is None else history_facts
    history = firm_history_through(source, year)
    top_ids = set(top_n_ids(history, rank=rank))
    year_rows = facts.loc[facts["analysis_year"] == year].copy()
    year_rows = year_rows.merge(
        history[["prior_awards", "prior_dollars"]],
        left_on="organization_id",
        right_index=True,
        how="left",
    )
    year_rows["prior_awards"] = year_rows["prior_awards"].fillna(0).astype(int)
    year_rows["prior_dollars"] = year_rows["prior_dollars"].fillna(0.0)
    year_rows["is_entrant"] = year_rows["prior_awards"].eq(0)
    year_rows["is_top10"] = year_rows["organization_id"].isin(top_ids)
    year_rows["bucket"] = [
        volume_bucket(p, t) for p, t in zip(year_rows["prior_awards"], year_rows["is_top10"], strict=True)
    ]
    year_rows["is_low_volume"] = year_rows["prior_awards"].between(1, LOW_VOLUME_MAX) & ~year_rows["is_top10"]
    year_rows["reliable_entrant_5y"] = year_rows["is_entrant"] & (year >= COVERAGE_START + LOOKBACK_ALT)
    year_rows["reliable_entrant_10y"] = year_rows["is_entrant"] & (
        year >= COVERAGE_START + LOOKBACK_PRIMARY
    )
    year_rows["incumbent_top10_ids"] = [tuple(sorted(top_ids))] * len(year_rows)
    return year_rows


def annotate_span(facts: pd.DataFrame, years, **kwargs) -> pd.DataFrame:
    pieces = [annotate_year(facts, year, **kwargs) for year in years]
    return pd.concat(pieces, ignore_index=True) if pieces else facts.iloc[0:0].copy()


history_years = range(COVERAGE_START, max(HEADLINE_YEARS) + 1)
working = frame.loc[frame["analysis_year"].between(COVERAGE_START, max(HEADLINE_YEARS))].copy()
labeled = annotate_span(working, history_years)
headline = labeled.loc[labeled["analysis_year"].isin(HEADLINE_YEARS)].copy()
phase1 = headline.loc[headline["phase_n"] == "I"].copy()
phase2 = headline.loc[headline["phase_n"] == "II"].copy()
len(labeled), len(headline), len(phase1), len(phase2)


In [ ]:
def pool_metrics(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    records = []
    if df.empty:
        return pd.DataFrame()
    for key, group in df.groupby(keys, dropna=False):
        key = (key,) if not isinstance(key, tuple) else key
        n = len(group)
        dollars = float(group["award_amount"].sum(min_count=1) or 0.0)
        firms = group["organization_id"].nunique()
        top = group.loc[group["is_top10"]]
        ent = group.loc[group["is_entrant"]]
        low = group.loc[group["is_low_volume"]]
        top_firms = top["organization_id"].nunique()
        top_awards = len(top)
        extra_slots = max(0, top_awards - top_firms)
        top_repeat = (
            top.groupby("organization_id").size().pipe(lambda s: int((s - 1).clip(lower=0).sum()))
        )
        top_repeat_dollars = 0.0
        if not top.empty:
            first_amt = top.sort_values("award_key").groupby("organization_id")["award_amount"].first()
            top_repeat_dollars = float(top["award_amount"].sum() - first_amt.sum())
        ent_med = float(ent["award_amount"].median()) if not ent.empty else None
        rec = {k: v for k, v in zip(keys, key, strict=True)}
        rec.update(
            {
                "awards": n,
                "dollars": dollars,
                "firms": firms,
                "top10_firms_active": top_firms,
                "top10_awards": top_awards,
                "top10_dollars": float(top["award_amount"].sum(min_count=1) or 0.0),
                "top10_award_share": top_awards / n if n else None,
                "top10_dollar_share": (float(top["award_amount"].sum()) / dollars) if dollars else None,
                "entrant_firms": ent["organization_id"].nunique(),
                "entrant_awards": len(ent),
                "entrant_dollars": float(ent["award_amount"].sum(min_count=1) or 0.0),
                "entrant_firm_share": ent["organization_id"].nunique() / firms if firms else None,
                "entrant_award_share": len(ent) / n if n else None,
                "entrant_dollar_share": (float(ent["award_amount"].sum()) / dollars) if dollars else None,
                "low_volume_firms": low["organization_id"].nunique(),
                "low_volume_awards": len(low),
                "low_volume_award_share": len(low) / n if n else None,
                "additional_firm_equivalent_slots": extra_slots,
                "top10_repeat_awards": top_repeat,
                "top10_repeat_dollars": top_repeat_dollars,
                "median_entrant_award": ent_med,
                "entrant_equivalent_capacity": (
                    (float(top["award_amount"].sum()) / ent_med) if ent_med and ent_med > 0 else None
                ),
                "entrant_equivalent_repeat_capacity": (
                    (top_repeat_dollars / ent_med) if ent_med and ent_med > 0 else None
                ),
                "award_count_hhi": _hhi(group.groupby("organization_id")["award_key"].nunique()),
                "dollar_hhi": _hhi(group.groupby("organization_id")["award_amount"].sum()),
            }
        )
        records.append(rec)
    return pd.DataFrame.from_records(records)


national_p1 = pool_metrics(phase1, ["analysis_year"])
agency_p1 = pool_metrics(phase1, ["agency", "analysis_year"])
national_p2 = pool_metrics(phase2, ["analysis_year"])
print("national Phase I by year")
national_p1


## 1. Incumbent top ten (award-count ranking, no look-ahead)

Each year's list is the ten firms with the most *prior* awards (all phases, all agencies)
through *t−1*. Display names are the baseline representative-name helper. Identity method
is the firm's modal method on prior awards.


In [ ]:
def yearly_top10_table(facts: pd.DataFrame, years) -> pd.DataFrame:
    rows = []
    for year in years:
        hist = firm_history_through(facts, year)
        ids = top_n_ids(hist)
        for rank, org in enumerate(ids, start=1):
            rec = hist.loc[org]
            rows.append(
                {
                    "year": year,
                    "rank": rank,
                    "organization_id": org,
                    "organization_name": rec["organization_name"],
                    "identity_method": rec["identity_method"],
                    "exact_identity": bool(rec["exact_identity"]),
                    "prior_awards": int(rec["prior_awards"]),
                    "prior_dollars": float(rec["prior_dollars"]),
                }
            )
    return pd.DataFrame(rows)


top10_by_year = yearly_top10_table(working, HEADLINE_YEARS)
stability = (
    top10_by_year.groupby(["organization_id", "organization_name", "identity_method"])
    .size()
    .rename("years_in_top10")
    .reset_index()
    .sort_values(["years_in_top10", "organization_name"], ascending=[False, True])
)
print("Ranking stability FY2021–FY2025 (how many headline years the firm is an incumbent top-10)")
display_df = stability
stability


In [ ]:
print("Incumbent top 10 entering FY2025 (ranked on awards through FY2024)")
top10_by_year.loc[top10_by_year["year"] == 2025]


## 2. Headline table — Phase I, FY2021–FY2025

Program-wide incumbent top 10 (not agency-specific). New entrant = zero prior awards
in this corpus. Low-volume incumbent = 1–5 prior awards and not in the top 10.
Additional-firm-equivalent slots = top-10 Phase I awards minus distinct top-10 firms
with a Phase I award in that year (mechanical upper bound).


In [ ]:
def summarize_slice(df: pd.DataFrame, label: str) -> pd.Series:
    n = len(df)
    firms = df["organization_id"].nunique()
    top = df.loc[df["is_top10"]]
    ent = df.loc[df["is_entrant"]]
    low = df.loc[df["is_low_volume"]]
    top_firms = top["organization_id"].nunique()
    return pd.Series(
        {
            "slice": label,
            "phase_i_awards": n,
            "phase_i_dollars": float(df["award_amount"].sum(min_count=1) or 0),
            "top10_awards": len(top),
            "top10_award_share": len(top) / n if n else None,
            "top10_firms_active": top_firms,
            "entrant_awards": len(ent),
            "entrant_firms": ent["organization_id"].nunique(),
            "entrant_award_share": len(ent) / n if n else None,
            "entrant_firm_share": ent["organization_id"].nunique() / firms if firms else None,
            "low_volume_awards": len(low),
            "low_volume_firms": low["organization_id"].nunique(),
            "low_volume_award_share": len(low) / n if n else None,
            "additional_firm_equivalent_slots": max(0, len(top) - top_firms),
        }
    )


headline_rows = [summarize_slice(phase1, "FY2021–FY2025 Phase I")]
headline_rows.extend(
    summarize_slice(phase1.loc[phase1["analysis_year"] == year], f"FY{year} Phase I")
    for year in HEADLINE_YEARS
)
headline_table = pd.DataFrame(headline_rows)
headline_table


## 3. Agency comparison (Phase I, FY2021–FY2025)


In [ ]:
agency_window = pool_metrics(phase1, ["agency"])
agency_window = agency_window.sort_values("top10_award_share", ascending=False)
agency_compare = agency_window[
    [
        "agency",
        "awards",
        "top10_award_share",
        "entrant_award_share",
        "entrant_firm_share",
        "low_volume_award_share",
        "additional_firm_equivalent_slots",
    ]
].copy()
agency_compare


## 4. Time series — top-10 Phase I award share vs entrant shares

Trend years: 1993–2025 so the 10-year lookback is available. Table only (matplotlib is
not in the core extra; re-run with `make install-ml` to plot).


In [ ]:
trend = labeled.loc[
    (labeled["phase_n"] == "I") & (labeled["analysis_year"].between(1993, 2025))
]
trend_national = pool_metrics(trend, ["analysis_year"]).sort_values("analysis_year")
trend_view = trend_national[
    [
        "analysis_year",
        "awards",
        "top10_award_share",
        "entrant_firm_share",
        "entrant_award_share",
        "low_volume_award_share",
        "additional_firm_equivalent_slots",
    ]
]
try:
    import matplotlib.pyplot as plt

    ax = trend_view.set_index("analysis_year")[
        ["top10_award_share", "entrant_award_share", "entrant_firm_share", "low_volume_award_share"]
    ].plot(figsize=(9, 4), title="Phase I shares, program-wide incumbent top 10")
    ax.set_ylabel("share")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; shares table follows.")
trend_view


## 5. Prior-award distribution (Phase I, FY2021–FY2025)

Buckets are mutually exclusive: entrant (0 prior), 1, 2–5, 6–20, 21+ outside the
incumbent top 10, and the incumbent top 10.


In [ ]:
BUCKET_ORDER = ["entrant", "1", "2-5", "6-20", "21+", "top10"]


def bucket_table(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    dollars = float(df["award_amount"].sum(min_count=1) or 0)
    rows = []
    for bucket in BUCKET_ORDER:
        part = df.loc[df["bucket"] == bucket]
        rows.append(
            {
                "bucket": bucket,
                "firms": part["organization_id"].nunique(),
                "awards": len(part),
                "award_share": len(part) / n if n else None,
                "dollars": float(part["award_amount"].sum(min_count=1) or 0),
                "dollar_share": (float(part["award_amount"].sum()) / dollars) if dollars else None,
            }
        )
    return pd.DataFrame(rows)


print("Phase I FY2021–FY2025")
bucket_table(phase1)


## 6. Crowd-out proxies (not a score, not causal)

1. **Additional-firm-equivalent award slots** — `top10_awards − top10_firms_active` on
   Phase I. Upper bound if each active top-10 firm kept one Phase I award and the rest
   became one-award slots. Those slots would not necessarily go to entrants.
2. **Entrant-equivalent funding capacity** — top-10 Phase I dollars ÷ median Phase I
   award to new entrants in the *same agency-year*. Repeat-only variant uses dollars
   after each top-10 firm's first Phase I award in that pool.
3. **Awards above the agency's historical low-concentration benchmark** — 25th percentile
   of that agency's annual top-10 Phase I award share over 1993–2020. For 2021–2025,
   excess = `max(0, share − p25) × awards`. Not "awards taken from" anyone.
4. **Association** — Spearman rank correlation of `top10_award_share` with
   `entrant_award_share` across agency-years with at least `MIN_POOL` Phase I awards.


In [ ]:
from scipy import stats

slots_national = int(headline_table.loc[headline_table["slice"] == "FY2021–FY2025 Phase I", "additional_firm_equivalent_slots"].iloc[0])
slots_agency_sum = int(agency_p1.loc[agency_p1["analysis_year"].isin(HEADLINE_YEARS), "additional_firm_equivalent_slots"].sum()) if not agency_p1.empty else 0

hist_agency = labeled.loc[
    (labeled["phase_n"] == "I") & (labeled["analysis_year"].between(1993, 2020))
]
hist_agency_m = pool_metrics(hist_agency, ["agency", "analysis_year"])
p25 = hist_agency_m.groupby("agency")["top10_award_share"].quantile(0.25).rename("p25_top10_share")
head_agency = agency_p1.loc[agency_p1["analysis_year"].isin(HEADLINE_YEARS)].merge(
    p25, on="agency", how="left"
)
head_agency["excess_share"] = (head_agency["top10_award_share"] - head_agency["p25_top10_share"]).clip(lower=0)
head_agency["awards_above_low_concentration_benchmark"] = (
    head_agency["excess_share"] * head_agency["awards"]
)
benchmark_excess = float(head_agency["awards_above_low_concentration_benchmark"].sum())

# dollar capacities: sum agency-year estimates in the headline window
cap = agency_p1.loc[agency_p1["analysis_year"].isin(HEADLINE_YEARS)]
entrant_equiv = float(cap["entrant_equivalent_capacity"].sum(min_count=1) or 0)
entrant_equiv_repeat = float(cap["entrant_equivalent_repeat_capacity"].sum(min_count=1) or 0)

panel = cap.loc[cap["awards"] >= MIN_POOL].copy()
if len(panel) >= 8:
    corr, pval = stats.spearmanr(panel["top10_award_share"], panel["entrant_award_share"])
    corr_firm, pval_firm = stats.spearmanr(panel["top10_award_share"], panel["entrant_firm_share"])
else:
    corr = pval = corr_firm = pval_firm = None

proxy = pd.DataFrame(
    {
        "proxy": [
            "additional-firm-equivalent slots (program-wide top 10, Phase I window)",
            "same slots summed over agency-year Phase I pools",
            "entrant-equivalent funding capacity (agency-year, all top-10 $)",
            "entrant-equivalent funding capacity (agency-year, repeat $ only)",
            "Phase I awards above agency 1993–2020 p25 top-10 share",
            "Spearman top10_award_share vs entrant_award_share (agency-years, n≥30)",
        ],
        "value": [
            slots_national,
            slots_agency_sum,
            round(entrant_equiv, 1),
            round(entrant_equiv_repeat, 1),
            round(benchmark_excess, 1),
            None if corr is None else f"{corr:.3f} (p={pval:.3f}, N={len(panel)})",
        ],
    }
)
proxy


## 7. Within-opportunity slice (agency × solicitation × phase)

Only rows with a non-empty solicitation number. Coverage is reported; do not generalize
the covered subset to the whole program. This still does not prove two firms bid against
each other — shared solicitation IDs can span multiple topics.


In [ ]:
p1_sol = phase1.loc[phase1["has_solicitation"]].copy()
coverage_sol = pd.Series(
    {
        "phase_i_headline_awards": len(phase1),
        "with_solicitation": len(p1_sol),
        "solicitation_coverage": len(p1_sol) / len(phase1) if len(phase1) else None,
        "distinct_solicitations": p1_sol.groupby(["agency", "solicitation"]).ngroups,
    }
)
opp = pool_metrics(p1_sol, ["agency", "solicitation"])
opp_multi = opp.loc[opp["awards"] >= 2].copy()
opp_summary = pd.Series(
    {
        "pools_with_2plus_awards": len(opp_multi),
        "mean_top10_award_share": float(opp_multi["top10_award_share"].mean()) if len(opp_multi) else None,
        "mean_entrant_award_share": float(opp_multi["entrant_award_share"].mean()) if len(opp_multi) else None,
        "mean_awards_per_pool": float(opp_multi["awards"].mean()) if len(opp_multi) else None,
        "pools_with_top10_repeat": int((opp_multi["top10_repeat_awards"] > 0).sum()) if len(opp_multi) else 0,
        "sum_top10_repeat_awards": int(opp_multi["top10_repeat_awards"].sum()) if len(opp_multi) else 0,
    }
)
pd.concat([coverage_sol.rename("coverage"), opp_summary.rename("opportunity_pools_2plus")], axis=1)


## 8. Sensitivities


In [ ]:
def headline_for(df: pd.DataFrame, label: str) -> pd.Series:
    return summarize_slice(df, label)


# 5y vs 10y lookback does not change prior_awards==0 in 2021–2025 (coverage starts 1983).
# Report the share of Phase I "entrants" that fail each reliability flag (should be ~0).
lookback = pd.Series(
    {
        "phase_i_zero_prior": int(phase1["is_entrant"].sum()),
        "unreliable_if_5y_required": int((phase1["is_entrant"] & ~phase1["reliable_entrant_5y"]).sum()),
        "unreliable_if_10y_required": int((phase1["is_entrant"] & ~phase1["reliable_entrant_10y"]).sum()),
    }
)

# Dollar-ranked top 10
labeled_dollars = annotate_span(working, HEADLINE_YEARS, rank="prior_dollars")
p1_dollars = labeled_dollars.loc[
    labeled_dollars["analysis_year"].isin(HEADLINE_YEARS) & (labeled_dollars["phase_n"] == "I")
]

# Exact identity only
exact_working = working.loc[working["exact_identity"]].copy()
labeled_exact = annotate_span(exact_working, HEADLINE_YEARS)
p1_exact = labeled_exact.loc[
    labeled_exact["analysis_year"].isin(HEADLINE_YEARS) & (labeled_exact["phase_n"] == "I")
]

# Agency-specific top 10: rank within each agency's own history
agency_specific_parts = []
for agency, sub in working.groupby("agency"):
    agency_specific_parts.append(annotate_span(sub, HEADLINE_YEARS))
labeled_agency_rank = pd.concat(agency_specific_parts, ignore_index=True)
p1_agency_rank = labeled_agency_rank.loc[
    labeled_agency_rank["analysis_year"].isin(HEADLINE_YEARS)
    & (labeled_agency_rank["phase_n"] == "I")
]

sens = pd.DataFrame(
    [
        headline_for(phase1, "count-ranked, program-wide, Phase I"),
        headline_for(p1_dollars, "dollar-ranked, program-wide, Phase I"),
        headline_for(p1_exact, "count-ranked, exact identity only, Phase I"),
        headline_for(p1_agency_rank, "count-ranked, agency-specific top 10, Phase I"),
        headline_for(phase2, "count-ranked, program-wide, Phase II"),
    ]
)
print("Lookback flags on headline Phase I (should be zero unreliable)")
print(lookback)
sens


## Appendix — retrospective top ten through FY2025 (look-ahead, not the headline ranking)

This list uses the full FY1983–FY2025 history. It **leaks future awards** into the ranking
and is only a descriptive roster of frequent winners.


In [ ]:
retro = working.loc[working["analysis_year"] <= 2025]
retro_hist = firm_history_through(retro, 2026)
retro_top = retro_hist.loc[top_n_ids(retro_hist)].reset_index().rename(columns={"index": "organization_id"})
retro_top.insert(0, "rank", range(1, len(retro_top) + 1))
retro_top[["rank", "organization_id", "organization_name", "identity_method", "prior_awards", "prior_dollars"]]


### Low-volume thresholds (1 / 1–3 / 1–5 / 1–10 prior awards)

Main definition remains 1–5. These rows exclude the incumbent top 10.


In [ ]:
def low_volume_at(df, lo, hi):
    mask = df["prior_awards"].between(lo, hi) & ~df["is_top10"]
    n = len(df)
    part = df.loc[mask]
    return {
        "prior_awards": f"{lo}" if lo == hi else f"{lo}–{hi}",
        "firms": part["organization_id"].nunique(),
        "awards": len(part),
        "award_share": len(part) / n if n else None,
    }

pd.DataFrame(
    [
        low_volume_at(phase1, 1, 1),
        low_volume_at(phase1, 1, 3),
        low_volume_at(phase1, 1, 5),
        low_volume_at(phase1, 1, 10),
    ]
)


### DoD component (Branch), Phase I FY2021–FY2025

Program-wide incumbent top 10, restricted to Department of Defense awards.


In [ ]:
dod = phase1.loc[phase1["agency"].eq("Department of Defense")].copy()
dod["branch"] = dod["branch"].fillna("(blank)").astype(str).str.strip().replace("", "(blank)")
dod_comp = pool_metrics(dod, ["branch"]).sort_values("awards", ascending=False)
dod_comp[
    ["branch", "awards", "top10_award_share", "entrant_award_share", "low_volume_award_share", "additional_firm_equivalent_slots"]
]


## Findings and caveats

| Claim | Evidence/artifact | Caveat or alternative explanation |
|---|---|---|
| Frequent winners absorb a measurable Phase I slot surplus | additional-firm-equivalent slots | Mechanical identity: extra awards to the same ten firms. Does not show that an excluded applicant existed or would have won. |
| Entrant participation can be compared with concentration | agency-year shares, Spearman | Association only. Demand, topic mix, budget, and open-topic policy all move both series. |
| Within-solicitation repeats are a tighter proxy | opportunity-pool table | Solicitation IDs are incomplete historically; even when present they are not bid-level competition. |
| CET choke-point overlay | not computed | Corpus-wide CET classifications are not on this checkout. |

**Not in the data:** proposals, evaluations, or losing applicants. Without that file this
notebook cannot identify causal crowd-out.

**Identity:** UEI/DUNS exact keys cover most recent awards; name-only keys can split or
merge firms (subsidiaries, aliases). The exact-identity sensitivity is the check. No
ad hoc name merges.

**Promotion:** exploratory. Not a study, not citable, not an A-CP7 Status upgrade.
